# Week 05 Practice: Change Data Capture with Debezium + Kafka + Iceberg

Five progressive parts:

1. **PostgreSQL + Debezium setup** — create source tables, register the CDC connector
2. **Exploring CDC events** — inspect the Debezium envelope (before/after/op)
3. **Bronze layer** — append-only Iceberg table storing every CDC event
4. **Silver layer** — MERGE INTO for current-state mirror
5. **Schema evolution** — ALTER TABLE at source, propagate through the pipeline

Run all cells **in order**. Each section builds on the previous one.

**Links:**
- MinIO Console: [http://localhost:9001](http://localhost:9001) (admin / password)
- Kafka Connect API: [http://localhost:8083](http://localhost:8083)
- Spark UI: [http://localhost:4040](http://localhost:4040)

---

**Reference:** *Designing Data-Intensive Applications*, 2nd Ed. — Ch. 3, 5, 12, 13

---
## Setup

The Spark session connects to Kafka, MinIO (S3-compatible), and Iceberg catalog.

In [33]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

requests + psycopg2 + kafka-python-ng ready


In [34]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
import os, time, json, requests

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3a://warehouse"
BOOTSTRAP   = "kafka:9092"

spark = (
    SparkSession.builder
    .appName("bdm_week05_cdc")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "hadoop")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")
print(f"Kafka: {BOOTSTRAP}")
print(f"S3:    {S3_ENDPOINT}")
print("Ready!")

Spark 4.1.0
Kafka: kafka:9092
S3:    http://minio:9000
Ready!


In [35]:
import psycopg2

PG_CONN = dict(host="postgres", port=5432, dbname="sourcedb", user="cdc_user", password="cdc_pass")

def pg_execute(sql, fetch=False):
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

ver = pg_execute("SELECT version();", fetch=True)
print(f"PostgreSQL: {ver[0][0][:60]}...")

wal = pg_execute("SHOW wal_level;", fetch=True)
print(f"wal_level = {wal[0][0]}")
assert wal[0][0] == "logical", "wal_level must be 'logical' for CDC!"

PostgreSQL: PostgreSQL 16.13 on aarch64-unknown-linux-musl, compiled by ...
wal_level = logical


---
## Part 1 — PostgreSQL Source + Debezium Connector

### 1.1 Create source tables in PostgreSQL

In [36]:
pg_execute("""
    DROP TABLE IF EXISTS customers;
    CREATE TABLE customers (
        id       SERIAL PRIMARY KEY,
        name     VARCHAR(100) NOT NULL,
        email    VARCHAR(200) NOT NULL,
        country  VARCHAR(50)  NOT NULL,
        created_at TIMESTAMP DEFAULT NOW()
    );
""")

pg_execute("""
    INSERT INTO customers (name, email, country) VALUES
        ('Alice Mets',    'alice@example.com',    'Estonia'),
        ('Bob Virtanen',  'bob@example.com',      'Finland'),
        ('Carol Ozols',   'carol@example.com',    'Latvia'),
        ('David Jonaitis','david@example.com',    'Lithuania'),
        ('Eva Svensson',  'eva@example.com',      'Sweden');
""")

rows = pg_execute("SELECT * FROM customers ORDER BY id;", fetch=True)
print(f"Inserted {len(rows)} customers:")
for r in rows:
    print(f"  {r}")

Inserted 5 customers:
  (1, 'Alice Mets', 'alice@example.com', 'Estonia', datetime.datetime(2026, 4, 6, 17, 20, 43, 884380))
  (2, 'Bob Virtanen', 'bob@example.com', 'Finland', datetime.datetime(2026, 4, 6, 17, 20, 43, 884380))
  (3, 'Carol Ozols', 'carol@example.com', 'Latvia', datetime.datetime(2026, 4, 6, 17, 20, 43, 884380))
  (4, 'David Jonaitis', 'david@example.com', 'Lithuania', datetime.datetime(2026, 4, 6, 17, 20, 43, 884380))
  (5, 'Eva Svensson', 'eva@example.com', 'Sweden', datetime.datetime(2026, 4, 6, 17, 20, 43, 884380))


### 1.2 Register the Debezium PostgreSQL connector

Debezium runs inside Kafka Connect. We POST configuration to the Connect REST API.

> **Note:** We set `value.converter.schemas.enable = false` to get cleaner JSON, but Debezium's
> default converter still wraps the message in a `{schema: ..., payload: ...}` envelope.
> The actual data lives under the `payload` key.

In [37]:
CONNECT_URL = "http://connect:8083"

for i in range(30):
    try:
        r = requests.get(f"{CONNECT_URL}/")
        if r.status_code == 200:
            print(f"Kafka Connect is ready: {r.json()}")
            break
    except:
        pass
    print(f"Waiting for Kafka Connect... ({i+1})")
    time.sleep(3)
else:
    raise RuntimeError("Kafka Connect did not start in time!")

Kafka Connect is ready: {'version': '3.9.0', 'commit': 'a60e31147e6b01ee', 'kafka_cluster_id': 'MkU3OEVBNTcwNTJENDM2Qg'}


In [38]:
requests.delete(f"{CONNECT_URL}/connectors/pg-cdc-connector")
time.sleep(2)

connector_config = {
    "name": "pg-cdc-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": "cdc_user",
        "database.password": "cdc_pass",
        "database.dbname": "sourcedb",
        "topic.prefix": "dbserver1",
        "table.include.list": "public.customers",
        "plugin.name": "pgoutput",
        "snapshot.mode": "initial",
        "key.converter.schemas.enable": "false",
        "value.converter.schemas.enable": "false",
    }
}

r = requests.post(
    f"{CONNECT_URL}/connectors",
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config),
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 201
{
  "name": "pg-cdc-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "cdc_user",
    "database.password": "cdc_pass",
    "database.dbname": "sourcedb",
    "topic.prefix": "dbserver1",
    "table.include.list": "public.customers",
    "plugin.name": "pgoutput",
    "snapshot.mode": "initial",
    "key.converter.schemas.enable": "false",
    "value.converter.schemas.enable": "false",
    "name": "pg-cdc-connector"
  },
  "tasks": [],
  "type": "source"
}


In [39]:
time.sleep(10)

r = requests.get(f"{CONNECT_URL}/connectors/pg-cdc-connector/status")
status = r.json()
print(f"Connector: {status['connector']['state']}")
for task in status.get('tasks', []):
    print(f"Task {task['id']}: {task['state']}")

assert status['connector']['state'] == 'RUNNING', f"Connector not running: {status}"

Connector: RUNNING
Task 0: RUNNING


In [40]:
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
topics = admin.list_topics()
admin.close()

print("Kafka topics:")
for t in sorted(topics):
    print(f"  {t}")

assert "dbserver1.public.customers" in topics, "CDC topic not found! Check connector status."
print("\n✓ CDC topic 'dbserver1.public.customers' exists!")

Kafka topics:
  __consumer_offsets
  _connect_configs
  _connect_offsets
  _connect_statuses
  dbserver1.public.customers

✓ CDC topic 'dbserver1.public.customers' exists!


---
## Part 2 — Exploring CDC Events

The initial snapshot produced `op: 'r'` (read) events for each existing row.
Let's consume them and inspect the Debezium envelope format.

> **Important:** Even with `schemas.enable=false`, the Debezium default JSON converter
> wraps messages in `{"schema": {...}, "payload": {...}}`. The actual CDC data is
> inside `payload`. We must extract from `$.payload.*` not from the top level.

In [41]:
# Read CDC events as a batch for inspection
cdc_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("key").cast("string").alias("key"),
        F.col("value").cast("string").alias("value"),
        "offset",
        "timestamp",
    )
)

print(f"CDC events in topic: {cdc_raw.count()}")
cdc_raw.show(5, truncate=80)

CDC events in topic: 21
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+-----------------------+
|                                                                             key|                                                                           value|offset|              timestamp|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+-----------------------+
|{"schema":{"type":"struct","fields":[{"type":"int32","optional":false,"defaul...|{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int3...|     0|2026-04-06 16:39:00.011|
|{"schema":{"type":"struct","fields":[{"type":"int32","optional":false,"defaul...|{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int3...|     1|2026-04-06 16:39:00.012|
|

In [42]:
# Pretty-print the first event — note the schema+payload wrapper
first_raw = cdc_raw.filter(F.col("value").isNotNull()).collect()[0]["value"]
first_event = json.loads(first_raw)
print("Top-level keys:", list(first_event.keys()))
print("\nPayload (the actual CDC data):")
print(json.dumps(first_event["payload"], indent=2))

Top-level keys: ['schema', 'payload']

Payload (the actual CDC data):
{
  "before": null,
  "after": {
    "id": 1,
    "name": "Alice Mets",
    "email": "alice@example.com",
    "country": "Estonia",
    "created_at": 1775493530483588
  },
  "source": {
    "version": "3.0.8.Final",
    "connector": "postgresql",
    "name": "dbserver1",
    "ts_ms": 1775493539381,
    "snapshot": "first",
    "db": "sourcedb",
    "sequence": "[null,\"26798760\"]",
    "ts_us": 1775493539381747,
    "ts_ns": 1775493539381747000,
    "schema": "public",
    "table": "customers",
    "txId": 746,
    "lsn": 26798760,
    "xmin": null
  },
  "transaction": null,
  "op": "r",
  "ts_ms": 1775493539446,
  "ts_us": 1775493539446801,
  "ts_ns": 1775493539446801297
}


In [44]:
def parse_cdc_events(kafka_df):
    """Parse Debezium CDC events from a RAW Kafka DataFrame (must have all Kafka columns)."""
    raw = kafka_df.select(
        F.col("offset").alias("kafka_offset"),
        F.col("partition").alias("kafka_partition"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").cast("string").alias("raw_value"),
    ).filter(F.col("raw_value").isNotNull())

    return raw.select(
        "kafka_offset", "kafka_partition", "kafka_timestamp",
        F.get_json_object("raw_value", "$.payload.op").alias("op"),
        F.get_json_object("raw_value", "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object("raw_value", "$.payload.before.name").alias("before_name"),
        F.get_json_object("raw_value", "$.payload.before.email").alias("before_email"),
        F.get_json_object("raw_value", "$.payload.before.country").alias("before_country"),
        F.get_json_object("raw_value", "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object("raw_value", "$.payload.after.name").alias("after_name"),
        F.get_json_object("raw_value", "$.payload.after.email").alias("after_email"),
        F.get_json_object("raw_value", "$.payload.after.country").alias("after_country"),
        F.get_json_object("raw_value", "$.payload.source.lsn").cast("long").alias("source_lsn"),
        F.get_json_object("raw_value", "$.payload.ts_ms").cast("long").alias("ts_ms"),
    )

# Read fresh from Kafka (NOT from cdc_raw which already dropped partition)
kafka_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

parsed = parse_cdc_events(kafka_raw)

print("Parsed CDC events (initial snapshot, op='r'):")
parsed.select("kafka_offset", "op", "after_id", "after_name", "after_email", "after_country", "ts_ms") \
      .orderBy("kafka_offset").show(truncate=False)

Parsed CDC events (initial snapshot, op='r'):
+------------+---+--------+--------------+------------------------+-------------+-------------+
|kafka_offset|op |after_id|after_name    |after_email             |after_country|ts_ms        |
+------------+---+--------+--------------+------------------------+-------------+-------------+
|0           |r  |1       |Alice Mets    |alice@example.com       |Estonia      |1775493539446|
|1           |r  |2       |Bob Virtanen  |bob@example.com         |Finland      |1775493539448|
|2           |r  |3       |Carol Ozols   |carol@example.com       |Latvia       |1775493539448|
|3           |r  |4       |David Jonaitis|david@example.com       |Lithuania    |1775493539448|
|4           |r  |5       |Eva Svensson  |eva@example.com         |Sweden       |1775493539448|
|5           |c  |1       |Alice Mets    |alice@example.com       |Estonia      |1775494113019|
|6           |c  |2       |Bob Virtanen  |bob@example.com         |Finland      |177549411

In [45]:
# Now make changes in PostgreSQL and observe the CDC events

# INSERT
pg_execute("INSERT INTO customers (name, email, country) VALUES ('Frank Muller', 'frank@example.com', 'Germany');")

# UPDATE
pg_execute("UPDATE customers SET email = 'alice.mets@newdomain.com' WHERE id = 1;")

# DELETE
pg_execute("DELETE FROM customers WHERE id = 3;")

print("Changes made: 1 INSERT, 1 UPDATE, 1 DELETE")
print("Waiting for Debezium to capture...")
time.sleep(5)

Changes made: 1 INSERT, 1 UPDATE, 1 DELETE
Waiting for Debezium to capture...


In [46]:
# Re-read the topic — should now have snapshot + change events
cdc_raw2 = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

parsed2 = parse_cdc_events(cdc_raw2)

print(f"Total CDC events (excl. tombstones): {parsed2.count()}")
print("\nr=snapshot read, c=create(INSERT), u=update, d=delete")
parsed2.select(
    "kafka_offset", "op",
    F.coalesce("after_id", "before_id").alias("id"),
    F.coalesce("after_name", "before_name").alias("name"),
    F.coalesce("after_email", "before_email").alias("email"),
    "ts_ms",
).orderBy("kafka_offset").show(truncate=False)

Total CDC events (excl. tombstones): 23

r=snapshot read, c=create(INSERT), u=update, d=delete
+------------+---+---+--------------+------------------------+-------------+
|kafka_offset|op |id |name          |email                   |ts_ms        |
+------------+---+---+--------------+------------------------+-------------+
|0           |r  |1  |Alice Mets    |alice@example.com       |1775493539446|
|1           |r  |2  |Bob Virtanen  |bob@example.com         |1775493539448|
|2           |r  |3  |Carol Ozols   |carol@example.com       |1775493539448|
|3           |r  |4  |David Jonaitis|david@example.com       |1775493539448|
|4           |r  |5  |Eva Svensson  |eva@example.com         |1775493539448|
|5           |c  |1  |Alice Mets    |alice@example.com       |1775494113019|
|6           |c  |2  |Bob Virtanen  |bob@example.com         |1775494113020|
|7           |c  |3  |Carol Ozols   |carol@example.com       |1775494113020|
|8           |c  |4  |David Jonaitis|david@example.com    

---
## Part 3 — Bronze Layer (Append-Only CDC Log)

The Bronze layer stores **every CDC event exactly as received** in an Iceberg table on MinIO.

- Append-only — we never update or delete rows in Bronze
- Preserves the complete history of all changes
- Our safety net for replay and debugging

In [47]:
# Create the Iceberg database and Bronze table
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.cdc")
spark.sql("USE lakehouse.cdc")

spark.sql("""
    CREATE OR REPLACE TABLE bronze_customers (
        kafka_offset    LONG,
        kafka_partition INT,
        kafka_timestamp TIMESTAMP,
        op              STRING,
        before_id       INT,
        before_name     STRING,
        before_email    STRING,
        before_country  STRING,
        after_id        INT,
        after_name      STRING,
        after_email     STRING,
        after_country   STRING,
        source_lsn      LONG,
        ts_ms           LONG
    ) USING iceberg
""")

print("Bronze table created on MinIO.")

Bronze table created on MinIO.


In [48]:
# Load all CDC events from Kafka into Bronze using our parser
kafka_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

bronze_df = parse_cdc_events(kafka_df)
bronze_df.writeTo("bronze_customers").append()

print(f"Bronze: {spark.table('bronze_customers').count()} CDC events loaded")
spark.sql("""
    SELECT kafka_offset, op, 
           COALESCE(after_id, before_id) AS id,
           COALESCE(after_name, before_name) AS name,
           COALESCE(after_email, before_email) AS email
    FROM bronze_customers
    ORDER BY kafka_offset
""").show(truncate=False)

Bronze: 23 CDC events loaded
+------------+---+---+--------------+------------------------+
|kafka_offset|op |id |name          |email                   |
+------------+---+---+--------------+------------------------+
|0           |r  |1  |Alice Mets    |alice@example.com       |
|1           |r  |2  |Bob Virtanen  |bob@example.com         |
|2           |r  |3  |Carol Ozols   |carol@example.com       |
|3           |r  |4  |David Jonaitis|david@example.com       |
|4           |r  |5  |Eva Svensson  |eva@example.com         |
|5           |c  |1  |Alice Mets    |alice@example.com       |
|6           |c  |2  |Bob Virtanen  |bob@example.com         |
|7           |c  |3  |Carol Ozols   |carol@example.com       |
|8           |c  |4  |David Jonaitis|david@example.com       |
|9           |c  |5  |Eva Svensson  |eva@example.com         |
|10          |c  |6  |Frank Muller  |frank@example.com       |
|11          |u  |1  |Alice Mets    |alice.mets@newdomain.com|
|12          |d  |3  |    

---
## Part 4 — Silver Layer (MERGE INTO — Current State)

The Silver layer mirrors the **current state** of the source PostgreSQL table.

We read from Bronze, deduplicate (keep latest event per primary key), and apply:
- `op = 'c'` or `'r'` → INSERT
- `op = 'u'` → UPDATE
- `op = 'd'` → DELETE

In [49]:
spark.sql("""
    CREATE OR REPLACE TABLE silver_customers (
        id       INT,
        name     STRING,
        email    STRING,
        country  STRING,
        updated_ts LONG
    ) USING iceberg
""")

print("Silver table created.")

Silver table created.


In [50]:
# Deduplicate Bronze: keep only the latest event per customer ID
spark.sql("""
    CREATE OR REPLACE TEMP VIEW cdc_latest AS
    SELECT * FROM (
        SELECT
            op,
            COALESCE(after_id, before_id) AS id,
            after_name AS name,
            after_email AS email,
            after_country AS country,
            ts_ms,
            ROW_NUMBER() OVER (
                PARTITION BY COALESCE(after_id, before_id)
                ORDER BY ts_ms DESC, kafka_offset DESC
            ) AS rn
        FROM bronze_customers
        WHERE op IS NOT NULL
    )
    WHERE rn = 1
""")

print("Deduplicated CDC events:")
spark.sql("SELECT op, id, name, email, country FROM cdc_latest ORDER BY id").show(truncate=False)

Deduplicated CDC events:
+---+---+--------------+------------------------+---------+
|op |id |name          |email                   |country  |
+---+---+--------------+------------------------+---------+
|u  |1  |Alice Mets    |alice.mets@newdomain.com|Estonia  |
|c  |2  |Bob Virtanen  |bob@example.com         |Finland  |
|d  |3  |NULL          |NULL                    |NULL     |
|c  |4  |David Jonaitis|david@example.com       |Lithuania|
|c  |5  |Eva Svensson  |eva@example.com         |Sweden   |
|c  |6  |Frank Muller  |frank@example.com       |Germany  |
|c  |7  |Grace Novak   |grace@example.com       |Poland   |
+---+---+--------------+------------------------+---------+



In [51]:
# Apply MERGE INTO — the core CDC pattern
spark.sql("""
    MERGE INTO silver_customers AS target
    USING cdc_latest AS source
    ON target.id = source.id

    WHEN MATCHED AND source.op = 'd' THEN DELETE

    WHEN MATCHED AND source.op IN ('u', 'c', 'r') THEN UPDATE SET
        name       = source.name,
        email      = source.email,
        country    = source.country,
        updated_ts = source.ts_ms

    WHEN NOT MATCHED AND source.op IN ('c', 'r') THEN INSERT
        (id, name, email, country, updated_ts)
        VALUES (source.id, source.name, source.email, source.country, source.ts_ms)
""")

print("MERGE INTO complete. Silver table (current state):")
spark.sql("SELECT * FROM silver_customers ORDER BY id").show(truncate=False)

MERGE INTO complete. Silver table (current state):
+---+--------------+-----------------+---------+-------------+
|id |name          |email            |country  |updated_ts   |
+---+--------------+-----------------+---------+-------------+
|2  |Bob Virtanen  |bob@example.com  |Finland  |1775496046152|
|4  |David Jonaitis|david@example.com|Lithuania|1775496046152|
|5  |Eva Svensson  |eva@example.com  |Sweden   |1775496046152|
|6  |Frank Muller  |frank@example.com|Germany  |1775496130746|
|7  |Grace Novak   |grace@example.com|Poland   |1775494139182|
+---+--------------+-----------------+---------+-------------+



In [52]:
# Compare with the actual PostgreSQL table — they should match!
pg_rows = pg_execute("SELECT id, name, email, country FROM customers ORDER BY id;", fetch=True)

print("PostgreSQL source (ground truth):")
for r in pg_rows:
    print(f"  {r}")

silver_count = spark.table('silver_customers').count()
print(f"\nSilver rows: {silver_count}")
print(f"PostgreSQL rows: {len(pg_rows)}")

if silver_count == len(pg_rows):
    print("\n✓ Match! CDC pipeline is working correctly.")
else:
    print("\n✗ Mismatch — check the MERGE logic.")

PostgreSQL source (ground truth):
  (1, 'Alice Mets', 'alice.mets@newdomain.com', 'Estonia')
  (2, 'Bob Virtanen', 'bob@example.com', 'Finland')
  (4, 'David Jonaitis', 'david@example.com', 'Lithuania')
  (5, 'Eva Svensson', 'eva@example.com', 'Sweden')
  (6, 'Frank Muller', 'frank@example.com', 'Germany')

Silver rows: 5
PostgreSQL rows: 5

✓ Match! CDC pipeline is working correctly.


### Verify idempotency — run MERGE again

MERGE INTO with a PK match is naturally idempotent. Running it again with the same data should produce no changes.

In [53]:
count_before = spark.table("silver_customers").count()

spark.sql("""
    MERGE INTO silver_customers AS target
    USING cdc_latest AS source
    ON target.id = source.id
    WHEN MATCHED AND source.op = 'd' THEN DELETE
    WHEN MATCHED AND source.op IN ('u', 'c', 'r') THEN UPDATE SET
        name = source.name, email = source.email, country = source.country, updated_ts = source.ts_ms
    WHEN NOT MATCHED AND source.op IN ('c', 'r') THEN INSERT
        (id, name, email, country, updated_ts)
        VALUES (source.id, source.name, source.email, source.country, source.ts_ms)
""")

count_after = spark.table("silver_customers").count()
print(f"Before: {count_before} rows")
print(f"After:  {count_after} rows")
print(f"\nIdempotent: {'✓ Yes' if count_before == count_after else '✗ No'} — safe to replay!")

Before: 5 rows
After:  5 rows

Idempotent: ✓ Yes — safe to replay!


---
## Part 5 — Schema Evolution

What happens when the source table schema changes? Let's add a column to PostgreSQL and observe how it propagates through the entire pipeline.

In [54]:
# Add a column to the PostgreSQL source table
pg_execute("ALTER TABLE customers ADD COLUMN phone VARCHAR(50);")
pg_execute("UPDATE customers SET phone = '+372-555-0001' WHERE id = 1;")
pg_execute("INSERT INTO customers (name, email, country, phone) VALUES ('Grace Novak', 'grace@example.com', 'Poland', '+48-555-0007');")

print("Schema changed: added 'phone' column, updated Alice, inserted Grace.")
print("Waiting for Debezium...")
time.sleep(5)

rows = pg_execute("SELECT id, name, email, country, phone FROM customers ORDER BY id;", fetch=True)
print("\nPostgreSQL now:")
for r in rows:
    print(f"  {r}")

Schema changed: added 'phone' column, updated Alice, inserted Grace.
Waiting for Debezium...

PostgreSQL now:
  (1, 'Alice Mets', 'alice.mets@newdomain.com', 'Estonia', '+372-555-0001')
  (2, 'Bob Virtanen', 'bob@example.com', 'Finland', None)
  (4, 'David Jonaitis', 'david@example.com', 'Lithuania', None)
  (5, 'Eva Svensson', 'eva@example.com', 'Sweden', None)
  (6, 'Frank Muller', 'frank@example.com', 'Germany', None)
  (7, 'Grace Novak', 'grace@example.com', 'Poland', '+48-555-0007')


In [55]:
# Read the new events from Kafka — the envelope now includes 'phone'
new_events_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("value"), "offset")
    .orderBy(F.desc("offset"))
    .limit(3)
)

print("Latest 3 events (should include 'phone'):")
for row in new_events_df.collect():
    val = row["value"]
    if val is None:
        print(f"  offset={row['offset']} [tombstone]")
        continue
    evt = json.loads(val)
    if "payload" in evt:
        evt = evt["payload"]
    after = evt.get("after") or {}
    print(f"  offset={row['offset']} op={evt.get('op')} phone={after.get('phone', 'N/A')} after={after}")

Latest 3 events (should include 'phone'):
  offset=26 op=c phone=+48-555-0007 after={'id': 7, 'name': 'Grace Novak', 'email': 'grace@example.com', 'country': 'Poland', 'created_at': 1775496165299887, 'phone': '+48-555-0007'}
  offset=25 op=u phone=+372-555-0001 after={'id': 1, 'name': 'Alice Mets', 'email': 'alice.mets@newdomain.com', 'country': 'Estonia', 'created_at': 1775496043884380, 'phone': '+372-555-0001'}
  offset=24 [tombstone]


In [56]:
# Evolve the Iceberg tables — metadata-only, no file rewrite!
spark.sql("ALTER TABLE bronze_customers ADD COLUMNS (before_phone STRING, after_phone STRING)")
spark.sql("ALTER TABLE silver_customers ADD COLUMNS (phone STRING)")

print("Schema evolved — 'phone' column added to Bronze and Silver.")
print("Old Parquet files on MinIO were NOT rewritten.")
print("Existing rows will return NULL for phone.")

Schema evolved — 'phone' column added to Bronze and Silver.
Old Parquet files on MinIO were NOT rewritten.
Existing rows will return NULL for phone.


In [57]:
# Helper: parse CDC events WITH phone field
def parse_cdc_events_v2(kafka_df):
    """Parse Debezium CDC events including the phone field (post schema evolution)."""
    raw = kafka_df.select(
        F.col("offset").alias("kafka_offset"),
        F.col("partition").alias("kafka_partition"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").cast("string").alias("raw_value"),
    ).filter(F.col("raw_value").isNotNull())

    return raw.select(
        "kafka_offset", "kafka_partition", "kafka_timestamp",
        F.get_json_object("raw_value", "$.payload.op").alias("op"),
        F.get_json_object("raw_value", "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object("raw_value", "$.payload.before.name").alias("before_name"),
        F.get_json_object("raw_value", "$.payload.before.email").alias("before_email"),
        F.get_json_object("raw_value", "$.payload.before.country").alias("before_country"),
        F.get_json_object("raw_value", "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object("raw_value", "$.payload.after.name").alias("after_name"),
        F.get_json_object("raw_value", "$.payload.after.email").alias("after_email"),
        F.get_json_object("raw_value", "$.payload.after.country").alias("after_country"),
        F.get_json_object("raw_value", "$.payload.source.lsn").cast("long").alias("source_lsn"),
        F.get_json_object("raw_value", "$.payload.ts_ms").cast("long").alias("ts_ms"),
        F.get_json_object("raw_value", "$.payload.before.phone").alias("before_phone"),
        F.get_json_object("raw_value", "$.payload.after.phone").alias("after_phone"),
    )


# Re-read ALL events from Kafka and reload Bronze
kafka_all = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

bronze_v2_df = parse_cdc_events_v2(kafka_all)

# Clear and reload Bronze
spark.sql("DELETE FROM bronze_customers WHERE 1=1")
bronze_v2_df.writeTo("bronze_customers").append()
print(f"Bronze reloaded: {spark.table('bronze_customers').count()} events")

spark.sql("""
    SELECT kafka_offset, op,
           COALESCE(after_id, before_id) AS id,
           COALESCE(after_name, before_name) AS name,
           after_phone AS phone
    FROM bronze_customers
    ORDER BY kafka_offset
""").show(truncate=False)

Bronze reloaded: 25 events
+------------+---+---+--------------+-------------+
|kafka_offset|op |id |name          |phone        |
+------------+---+---+--------------+-------------+
|0           |r  |1  |Alice Mets    |NULL         |
|1           |r  |2  |Bob Virtanen  |NULL         |
|2           |r  |3  |Carol Ozols   |NULL         |
|3           |r  |4  |David Jonaitis|NULL         |
|4           |r  |5  |Eva Svensson  |NULL         |
|5           |c  |1  |Alice Mets    |NULL         |
|6           |c  |2  |Bob Virtanen  |NULL         |
|7           |c  |3  |Carol Ozols   |NULL         |
|8           |c  |4  |David Jonaitis|NULL         |
|9           |c  |5  |Eva Svensson  |NULL         |
|10          |c  |6  |Frank Muller  |NULL         |
|11          |u  |1  |Alice Mets    |NULL         |
|12          |d  |3  |              |NULL         |
|14          |u  |1  |Alice Mets    |+372-555-0001|
|15          |c  |7  |Grace Novak   |+48-555-0007 |
|16          |c  |1  |Alice Mets    |

In [62]:
# Re-create dedup view with phone and MERGE into Silver
spark.sql("""
    CREATE OR REPLACE TEMP VIEW cdc_latest_v2 AS
    SELECT * FROM (
        SELECT
            op,
            COALESCE(after_id, before_id) AS id,
            after_name AS name,
            after_email AS email,
            after_country AS country,
            after_phone AS phone,
            ts_ms,
            ROW_NUMBER() OVER (
                PARTITION BY COALESCE(after_id, before_id)
                ORDER BY ts_ms DESC, kafka_offset DESC
            ) AS rn
        FROM bronze_customers
        WHERE op IS NOT NULL
    )
    WHERE rn = 1
""")

spark.sql("""
    MERGE INTO silver_customers AS target
    USING cdc_latest_v2 AS source
    ON target.id = source.id
    WHEN MATCHED AND source.op = 'd' THEN DELETE
    WHEN MATCHED AND source.op IN ('u', 'c', 'r') THEN UPDATE SET
        name = source.name, email = source.email, country = source.country,
        updated_ts = source.ts_ms, phone = source.phone
    WHEN NOT MATCHED AND source.op IN ('c', 'r', 'u') THEN INSERT
        (id, name, email, country, updated_ts, phone)
        VALUES (source.id, source.name, source.email, source.country, source.ts_ms, source.phone)
""")

print("Silver after schema evolution:")
spark.sql("SELECT * FROM silver_customers ORDER BY id").show(truncate=False)

Silver after schema evolution:
+---+--------------+------------------------+---------+-------------+-------------+
|id |name          |email                   |country  |updated_ts   |phone        |
+---+--------------+------------------------+---------+-------------+-------------+
|1  |Alice Mets    |alice.mets@newdomain.com|Estonia  |1775496165640|+372-555-0001|
|2  |Bob Virtanen  |bob@example.com         |Finland  |1775496046152|NULL         |
|4  |David Jonaitis|david@example.com       |Lithuania|1775496046152|NULL         |
|5  |Eva Svensson  |eva@example.com         |Sweden   |1775496046152|NULL         |
|6  |Frank Muller  |frank@example.com       |Germany  |1775496130746|NULL         |
|7  |Grace Novak   |grace@example.com       |Poland   |1775496165641|+48-555-0007 |
+---+--------------+------------------------+---------+-------------+-------------+



In [59]:
# Time travel — view Silver table history
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakehouse.cdc.silver_customers.snapshots
    ORDER BY committed_at
""").collect()

print("Silver snapshots:")
for s in snapshots:
    print(f"  {s['snapshot_id']}  {s['committed_at']}  {s['operation']}")

if len(snapshots) >= 2:
    old_snap = snapshots[0]['snapshot_id']
    print(f"\nTime travel to first snapshot ({old_snap}):")
    spark.sql(f"SELECT * FROM silver_customers VERSION AS OF {old_snap} ORDER BY id").show(truncate=False)

Silver snapshots:
  2512083657154761406  2026-04-06 16:48:58.209000  overwrite
  4332368268000501295  2026-04-06 16:48:58.628000  overwrite
  8012332854173265064  2026-04-06 17:01:28.305000  overwrite
  7926350527553692324  2026-04-06 17:03:52.560000  overwrite
  3348491623179580457  2026-04-06 17:22:39.350000  append
  3173003919244090059  2026-04-06 17:22:43.239000  overwrite
  8866654050404506646  2026-04-06 17:22:58.841000  overwrite

Time travel to first snapshot (2512083657154761406):
+---+----+-----+-------+----------+
|id |name|email|country|updated_ts|
+---+----+-----+-------+----------+
+---+----+-----+-------+----------+



---
## Summary

| Layer | Table | Rows | Description |
|-------|-------|------|-------------|
| **Bronze** | `bronze_customers` | Every CDC event | Append-only, raw Debezium events |
| **Silver** | `silver_customers` | Current state | MERGE INTO from Bronze, mirrors PostgreSQL |

In [61]:
spark.sql("SELECT * FROM silver_customers ORDER BY id").show(truncate=False)

+---+--------------+-----------------+---------+-------------+------------+
|id |name          |email            |country  |updated_ts   |phone       |
+---+--------------+-----------------+---------+-------------+------------+
|2  |Bob Virtanen  |bob@example.com  |Finland  |1775496046152|NULL        |
|4  |David Jonaitis|david@example.com|Lithuania|1775496046152|NULL        |
|5  |Eva Svensson  |eva@example.com  |Sweden   |1775496046152|NULL        |
|6  |Frank Muller  |frank@example.com|Germany  |1775496130746|NULL        |
|7  |Grace Novak   |grace@example.com|Poland   |1775496165641|+48-555-0007|
+---+--------------+-----------------+---------+-------------+------------+



In [60]:
print("=== CDC Pipeline Summary ===")
print(f"{'Layer':<10} {'Table':<25} {'Rows':>6}")
print("-" * 43)
for layer, table in [("Bronze", "bronze_customers"), ("Silver", "silver_customers")]:
    cnt = spark.table(table).count()
    print(f"{layer:<10} {table:<25} {cnt:>6}")

# Final validation
pg_rows = pg_execute("SELECT id, name, email, country FROM customers ORDER BY id;", fetch=True)
silver_count = spark.table('silver_customers').count()
print(f"\nPostgreSQL: {len(pg_rows)} rows | Silver: {silver_count} rows")
if silver_count == len(pg_rows):
    print("✓ Pipeline verified — Silver matches PostgreSQL!")

print("\nBrowse MinIO: http://localhost:9001 → warehouse → cdc.db/")
print("All tables have ACID transactions, snapshots, and time travel.")

=== CDC Pipeline Summary ===
Layer      Table                       Rows
-------------------------------------------
Bronze     bronze_customers              25
Silver     silver_customers               5

PostgreSQL: 6 rows | Silver: 5 rows

Browse MinIO: http://localhost:9001 → warehouse → cdc.db/
All tables have ACID transactions, snapshots, and time travel.


---
## Exercises

You are free to use any resource (documentation, AI tools, etc.).

---
### Exercise 1 — Snapshot rollback after bad data

1. Insert a "bad" customer into PostgreSQL: `INSERT INTO customers (name, email, country) VALUES ('BAD_DATA', 'bad@bad.com', 'INVALID');`
2. Wait 5 seconds for Debezium.
3. Reload Bronze and run MERGE into Silver.
4. Verify the bad row appears in Silver.
5. Find the Silver snapshot ID *before* the bad MERGE (from `silver_customers.snapshots`).
6. Use `CALL lakehouse.system.rollback_to_snapshot('cdc.silver_customers', <id>)` to undo it.
7. Verify Silver is clean again.

**Question:** Why is having the Bronze layer critical for recovery? What would happen with Pattern 2 (direct materialization) instead?

In [ ]:
# Exercise 1 — your answer


---
### Exercise 2 — Add a second source table

1. Create a new table in PostgreSQL: `orders (id SERIAL PRIMARY KEY, customer_id INT, product VARCHAR(100), amount DECIMAL(10,2), created_at TIMESTAMP DEFAULT NOW())`
2. Insert 5 sample orders.
3. Update the Debezium connector configuration to also include `public.orders`:
   ```python
   requests.put(
       f"{CONNECT_URL}/connectors/pg-cdc-connector/config",
       headers={"Content-Type": "application/json"},
       data=json.dumps({...config with table.include.list: "public.customers,public.orders"...}),
   )
   ```
4. Verify a new Kafka topic `dbserver1.public.orders` is created.
5. Create `bronze_orders` and `silver_orders` Iceberg tables.
6. Load Bronze, MERGE into Silver.
7. Make some changes (INSERT, UPDATE, DELETE) to orders in PostgreSQL, reload, and verify Silver is correct.

**Bonus:** Join `silver_customers` with `silver_orders` to show "customer name + their orders".

In [ ]:
# Exercise 2 — your answer


---
## Further reading

- [Debezium PostgreSQL Connector Docs](https://debezium.io/documentation/reference/stable/connectors/postgresql.html)
- [Debezium Tutorial](https://debezium.io/documentation/reference/stable/tutorial.html)
- [Apache Iceberg — MERGE INTO](https://iceberg.apache.org/docs/latest/spark-writes/#merge-into)
- [Iceberg Schema Evolution](https://iceberg.apache.org/docs/latest/evolution/)
- [Spark Structured Streaming + Kafka](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)
- [Ryft.io — CDC Strategies in Iceberg (2025)](https://www.ryft.io/blog/cdc-strategies-in-apache-iceberg)
- *Designing Data-Intensive Applications*, 2nd Ed. — Ch. 3, 5, 12, 13